In [ ]:
# pandas for loading and working with the CSV as a dataframe
import pandas as pd

# Task 2 gets its own notebook, so reload the raw data fresh —
# nothing carries over from 01_profile.ipynb's kernel.
df = pd.read_csv("../data/raw_pos_export.csv")

# Sanity check: should be (131, 17), matching Task 1's profiling.
df.shape

In [ ]:
# Fix: pandas inferred one format from the first rows and coerced everything
# else to NaT instead of trying each row's format individually.
# format="mixed" tells it to parse each row on its own terms.
df["ts_parsed"] = pd.to_datetime(df["transaction_ts"], format="mixed", errors="coerce")

# Should now be close to 1 (the single blank transaction_ts from Task 1),
# not 108 like the naive parse produced.
df["ts_parsed"].isna().sum()

In [ ]:
# Drop rows where parsing still failed (or was genuinely blank) so they don't
# poison the sort comparison the way the earlier 108 NaTs did.
valid = df.dropna(subset=["ts_parsed"]).copy()

# argsort() gives the row order each column would need to be sorted ascending.
# If row_id and ts_parsed produce the same order, row_id is a safe proxy
# for chronological order.
row_id_order = valid["row_id"].argsort()
ts_order = valid["ts_parsed"].argsort()

# Count of positions where the two orderings disagree.
# 0 = row_id is a perfectly reliable time-ordering proxy.
# Any non-zero count = row_id can drift from true chronological order,
# which matters for how tightly you can trust "nearby rows = nearby time"
# when you build the invoice reconstruction rule in Task 3.
(row_id_order.values != ts_order.values).sum()

In [ ]:
df